# Imports

In [ ]:
# Import neccessary

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()


# import pymc as pm
# import bambi as bmb
# import arviz as az
# import statsmodels.api as sm
import statsmodels.api as sm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data cleaning

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/[Group 27] Data 102 Final Project Q1/ozone-ca.csv")

In [ ]:
df_2011 = df[df['date'].str.contains('2011')]
df_2011.head()

,year,date,statefips,countyfips,ctfips,latitude,longitude,ds_o3_pred,ds_o3_stdd
0,2011,02JAN2011,6,6001,6001400100,37.86754,-122.23181,24.150185,4.455231
1,2011,02JAN2011,6,6001,6001400200,37.84817,-122.24948,24.176594,4.461640
2,2011,02JAN2011,6,6001,6001400300,37.84056,-122.25442,24.066095,4.394948
3,2011,02JAN2011,6,6001,6001400400,37.84801,-122.25752,24.092117,4.491825
4,2011,02JAN2011,6,6001,6001400500,37.84853,-122.26480,23.872848,4.425602


In [ ]:
df_county_daily = df_2011.groupby(["statefips", "countyfips", "date"])["ds_o3_pred"].mean().reset_index()

df_annual = df_county_daily.groupby(["statefips", "countyfips"])["ds_o3_pred"].mean().reset_index(name="ozone_mean_2011")

In [ ]:
import geopandas as gpd
shapefile_path = "/content/drive/MyDrive/[Group 27] Data 102 Final Project Q1/California_Counties/Counties.shp"
counties = gpd.read_file(shapefile_path)

In [ ]:
counties.head()

,Year,CountyName,COUNTY_FIP,DistrictCo,geometry
0,2023,Alameda,001,22,"MULTIPOLYGON (((-13577345.444 4504491.196, -13..."
1,2023,Alpine,003,2,"POLYGON ((-13312191.387 4680675.259, -13312135..."
2,2023,Amador,005,2,"POLYGON ((-13366398.979 4679185.227, -13366398..."
3,2023,Colusa,011,5,"POLYGON ((-13568735.664 4776807.645, -13568728..."
4,2023,El Dorado,017,16,"POLYGON ((-13347688.733 4712126.051, -13347699..."


In [ ]:
df_annual['countyfips'] = df_annual['countyfips'] % 6000

In [ ]:
counties['COUNTY_FIP'] = pd.to_numeric(counties['COUNTY_FIP'])
merged = counties.merge(df_annual, left_on="COUNTY_FIP", right_on="countyfips")

In [ ]:
df_pm = pd.read_csv("/content/drive/MyDrive/[Group 27] Data 102 Final Project Q1/pm25-ca.csv")
df_pm.head()

,year,date,statefips,countyfips,ctfips,latitude,longitude,ds_pm_pred,ds_pm_stdd
0,2011,17JAN2011,6,6001,6001400100,37.86754,-122.23181,21.602403,4.681299
1,2011,17JAN2011,6,6001,6001400200,37.84817,-122.24948,21.725637,4.715272
2,2011,17JAN2011,6,6001,6001400300,37.84056,-122.25442,21.928701,4.661169
3,2011,17JAN2011,6,6001,6001400400,37.84801,-122.25752,21.792893,4.851187
4,2011,17JAN2011,6,6001,6001400500,37.84853,-122.26480,21.577514,4.731135


In [ ]:
pm25_2011 = df_pm[df_pm['date'].str.contains('2011')]
pm25_county_daily = pm25_2011.groupby(["statefips", "countyfips", "date"])["ds_pm_pred"].mean().reset_index()


In [ ]:
pm25_annual = pm25_county_daily.groupby(["statefips", "countyfips"])["ds_pm_pred"].mean().reset_index(name="pm25_mean_2011")
pm25_annual['countyfips'] = pm25_annual['countyfips'] % 6000

In [ ]:
merged_pm25 = counties.merge(pm25_annual, left_on="COUNTY_FIP", right_on="countyfips")


In [ ]:
ca_copd = pd.read_csv("/content/drive/MyDrive/[Group 27] Data 102 Final Project Q1/copd_ca_counties.csv")

In [ ]:
ca_copd['LocationID'] = ca_copd['LocationID'] % 6000
ca_age_adjusted_prevalance = ca_copd[ca_copd['Data_Value_Type'] == 'Age-adjusted prevalence']
ca_age_adjusted_prevalance.head()

,Year,StateAbbr,StateDesc,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,...,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text,Geolocation
1,2022,CA,California,Inyo,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among ad...,%,Age-adjusted prevalence,5.7,...,5.1,6.3,"18,718","14,942",27,HLTHOUT,COPD,AgeAdjPrv,COPD,POINT (-117.411112581295 36.5114226868129)
4,2022,CA,California,Humboldt,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among ad...,%,Age-adjusted prevalence,7.2,...,6.4,8.0,"135,010","110,011",23,HLTHOUT,COPD,AgeAdjPrv,COPD,POINT (-123.875674190259 40.6992051844633)
8,2022,CA,California,Santa Clara,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among ad...,%,Age-adjusted prevalence,3.8,...,3.3,4.2,"1,870,945","1,490,785",85,HLTHOUT,COPD,AgeAdjPrv,COPD,POINT (-121.695251446392 37.2316875247769)
9,2022,CA,California,Lake,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among ad...,%,Age-adjusted prevalence,6.4,...,5.7,7.1,"68,191","53,338",33,HLTHOUT,COPD,AgeAdjPrv,COPD,POINT (-122.753413599125 39.0999886976907)
11,2022,CA,California,Kern,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among ad...,%,Age-adjusted prevalence,7.0,...,6.3,7.8,"916,108","655,754",29,HLTHOUT,COPD,AgeAdjPrv,COPD,POINT (-118.730029086317 35.3428262386385)


# Old models

## Linear regression

In [ ]:
merged_copd = counties.merge(ca_age_adjusted_prevalance, left_on="COUNTY_FIP", right_on="LocationID")

In [ ]:
merged_df = ca_age_adjusted_prevalance.merge(df_annual, left_on = "LocationID", right_on="countyfips").merge(pm25_annual, left_on = "LocationID", right_on="countyfips")
# merged_df.head()


In [ ]:
X = merged_df[["ozone_mean_2011", "pm25_mean_2011"]]
X = sm.add_constant(X)
y = merged_df["Data_Value"]

In [ ]:
model = sm.GLM(y, X, family=sm.families.Gaussian())
results = model.fit()

# print(results.summary())

## Gaussian GLM on Logit-Transformed Rate

In [ ]:
merged_df["copd_prop"] = merged_df.Data_Value / 100
merged_df["copd_logit"] = np.log(merged_df.copd_prop / (1 - merged_df.copd_prop))
y = merged_df["copd_logit"]

In [ ]:
logit_model = sm.GLM(y, X, family=sm.families.Gaussian())
results = logit_model.fit()

# print(results.summary())

In [ ]:
y = np.column_stack([
    merged_df.Data_Value, # successes per 100
    100 - merged_df.Data_Value # failures per 100
])

logistic_model = sm.GLM(y, X, family=sm.families.Binomial())
results = logistic_model.fit()

# print(results.summary())

## Random forests regressor

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def fit_and_predict(model_class, model_args, model_name, X_train, X_test, y_train, y_test):
    model = model_class(**model_args)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)


    print(f"Results for {model_name}:")
    print(f"MSE on test set: {mse}")
    print(f"MAE on test set: {mae}")
    print(f"R^2 on test set: {r2}")
    return model

In [ ]:
y = merged_df["Data_Value"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
fit_and_predict(
    RandomForestRegressor,
    {
        "n_estimators": 500, # many trees -> stable predictions
        "max_depth": 4, # prevents overfitting
        "min_samples_split": 4, # increases generalization
        "random_state": 42
    },
    "random forest",
    X_train, X_test, y_train, y_test
)

Results for random forest:
MSE on test set: 0.8138170392687211
MAE on test set: 0.6557498665454522
R^2 on test set: 0.47425533722196955


RandomForestRegressor(max_depth=4, min_samples_split=4, n_estimators=500,
                      random_state=42)

In [ ]:
# from sklearn.tree import DecisionTreeRegressor
# fit_and_predict(
#     DecisionTreeRegressor, {}, 'decision tree',
#     X_train, X_test, y_train, y_test
# );

In [ ]:
ca_counties_health = pd.read_csv("/content/drive/MyDrive/[Group 27] Data 102 Final Project Q1/ca_health.csv")

In [ ]:
ca_counties_health['CountyFIPS'] = ca_counties_health['CountyFIPS'] % 6000

## Linear regression with more predictors [calculuate AIC]

In [ ]:
merged_ca = merged_df.merge(ca_counties_health, left_on="LocationID", right_on="CountyFIPS")
merged_ca.columns

Index(['Year', 'StateAbbr', 'StateDesc', 'LocationName', 'DataSource',
       'Category', 'Measure', 'Data_Value_Unit', 'Data_Value_Type',
       'Data_Value', 'Data_Value_Footnote_Symbol', 'Data_Value_Footnote',
       'Low_Confidence_Limit', 'High_Confidence_Limit', 'TotalPopulation',
       'TotalPop18plus', 'LocationID', 'CategoryID', 'MeasureId',
       'DataValueTypeID', 'Short_Question_Text', 'Geolocation', 'statefips_x',
       'countyfips_x', 'ozone_mean_2011', 'statefips_y', 'countyfips_y',
       'pm25_mean_2011', 'copd_prop', 'copd_logit', 'County', 'CountyFIPS',
       'NoInsurance_AdjPrev', 'Asthma_AdjPrev', 'Obesity_AdjPrev',
       'PoorHealth_AdjPrev', 'Smoking_AdjPrev', 'Smoking_Adj95CI'],
      dtype='object')

In [ ]:
X = merged_ca[["ozone_mean_2011", "pm25_mean_2011",'NoInsurance_AdjPrev', 'Asthma_AdjPrev', 'Obesity_AdjPrev', 'PoorHealth_AdjPrev', 'Smoking_AdjPrev' ]]
X = sm.add_constant(X)
y = merged_ca["Data_Value"]

In [ ]:
model = sm.GLM(y, X, family=sm.families.Gaussian())
results = model.fit()

print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             Data_Value   No. Observations:                   58
Model:                            GLM   Df Residuals:                       50
Model Family:                Gaussian   Df Model:                            7
Link Function:               Identity   Scale:                         0.32153
Method:                          IRLS   Log-Likelihood:                -45.089
Date:                Mon, 15 Dec 2025   Deviance:                       16.077
Time:                        20:09:22   Pearson chi2:                     16.1
No. Iterations:                     3   Pseudo R-squ. (CS):             0.9619
Covariance Type:            nonrobust                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -2.7876    

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [ ]:
from sklearn.tree import DecisionTreeRegressor
fit_and_predict(
    DecisionTreeRegressor, {}, 'decision tree',
    X_train, X_test, y_train, y_test
);

Results for decision tree:
MSE on test set: 0.38611111111111107
MAE on test set: 0.4388888888888889
R^2 on test set: 0.7505632763742947


## Gaussian GLM on Logit-Transformed Rate

In [ ]:
y = merged_df["copd_logit"]
logit_model = sm.GLM(y, X, family=sm.families.Gaussian())
results = logit_model.fit()

print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             copd_logit   No. Observations:                   58
Model:                            GLM   Df Residuals:                       50
Model Family:                Gaussian   Df Model:                            7
Link Function:               Identity   Scale:                       0.0099836
Method:                          IRLS   Log-Likelihood:                 55.603
Date:                Mon, 15 Dec 2025   Deviance:                      0.49918
Time:                        20:09:22   Pearson chi2:                    0.499
No. Iterations:                     3   Pseudo R-squ. (CS):             0.9820
Covariance Type:            nonrobust                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -4.6506    

## Binomial GLM with Logistic Regression [compute AIC]

In [ ]:
y = np.column_stack([
    merged_df.Data_Value, # successes per 100
    100 - merged_df.Data_Value # failures per 100
])

logistic_model = sm.GLM(y, X, family=sm.families.Binomial())
results = logistic_model.fit()

print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           ['y1', 'y2']   No. Observations:                   58
Model:                            GLM   Df Residuals:                       50
Model Family:                Binomial   Df Model:                            7
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -104.66
Date:                Mon, 15 Dec 2025   Deviance:                       2.8887
Time:                        20:09:22   Pearson chi2:                     2.96
No. Iterations:                     6   Pseudo R-squ. (CS):             0.1780
Covariance Type:            nonrobust                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -4.6619    

## Random forest Regressor

In [ ]:
X = merged_ca[["ozone_mean_2011", "pm25_mean_2011",'NoInsurance_AdjPrev', 'Asthma_AdjPrev', 'Obesity_AdjPrev', 'PoorHealth_AdjPrev', 'Smoking_AdjPrev' ]]
y = merged_ca["Data_Value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [ ]:
fit_and_predict(
    RandomForestRegressor,
    {
        "n_estimators": 500, # many trees -> stable predictions
        "max_depth": 4, # prevents overfitting
        "min_samples_split": 4, # increases generalization
        "random_state": 42
    },
    "random forest",
    X_train, X_test, y_train, y_test
)

Results for random forest:
MSE on test set: 0.35494550477047937
MAE on test set: 0.41617999519386334
R^2 on test set: 0.770696980149472


RandomForestRegressor(max_depth=4, min_samples_split=4, n_estimators=500,
                      random_state=42)

# Linear Regression (minimized)

In [ ]:
X = merged_ca[["ozone_mean_2011", 'Asthma_AdjPrev', 'Smoking_AdjPrev' ]]
X = sm.add_constant(X)
y = merged_ca["Data_Value"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = sm.GLM(y_train, X_train, family=sm.families.Gaussian())
results = model.fit()
print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             Data_Value   No. Observations:                   40
Model:                            GLM   Df Residuals:                       36
Model Family:                Gaussian   Df Model:                            3
Link Function:               Identity   Scale:                         0.26903
Method:                          IRLS   Log-Likelihood:                -28.392
Date:                Mon, 15 Dec 2025   Deviance:                       9.6852
Time:                        20:09:23   Pearson chi2:                     9.69
No. Iterations:                     3   Pseudo R-squ. (CS):             0.9730
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -3.1231      1.390     

In [ ]:
# predictions:
y_pred = results.predict(X)

# true values:
y_true = y

mse = np.mean((y_true - y_pred)**2)
mae = np.mean(np.abs(y_true - y_pred))
ss_res = np.sum((y_true - y_pred)**2)
ss_tot = np.sum((y_true - np.mean(y_true))**2)
r2 = 1 - ss_res/ss_tot

print("MSE:", mse)
print("MAE:", mae)
print("R^2:", r2)

MSE: 0.28716285343428366
MAE: 0.3994631241630078
R^2: 0.7831146916163909


# Binomial GLM (Minimized)

In [ ]:
y = np.column_stack([
    merged_df.Data_Value, # successes per 100
    100 - merged_df.Data_Value # failures per 100
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = sm.GLM(y_train, X_train, family=sm.families.Binomial())
results = model.fit()
print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           ['y1', 'y2']   No. Observations:                   40
Model:                            GLM   Df Residuals:                       36
Model Family:                Binomial   Df Model:                            3
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -72.336
Date:                Mon, 15 Dec 2025   Deviance:                       1.8887
Time:                        20:09:23   Pearson chi2:                     1.90
No. Iterations:                     6   Pseudo R-squ. (CS):             0.1640
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -4.5807      1.217     

In [ ]:
# predicted probability on test set
pred_prob = results.predict(X_test) * 100 # multiply by 100 to convert to original scale

# true rate: successes / (successes + failures)
true_prob = y_test[:, 0] / (y_test[:, 0] + y_test[:, 1]) * 100 # multiply by 100 to convert to original scale

mse = np.mean((true_prob - pred_prob)**2)
mae = np.mean(np.abs(true_prob - pred_prob))

# R^2
ss_res = np.sum((true_prob - pred_prob)**2)
ss_tot = np.sum((true_prob - np.mean(true_prob))**2)
r2 = 1 - ss_res/ss_tot

print("MSE:", mse)
print("MAE:", mae)
print("R^2:", r2)

MSE: 0.4211739902526664
MAE: 0.4404530536904747
R^2: 0.7279118440734076
